In [5]:
import numpy as np
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

# ==========================================
# 1. SETUP DATASETS AND CLASSIFIER SCORES
# ==========================================
datasets = [
    "advert0", "advert1", "audiology", "boxing1", "boxing2", "breast", "car", "chess",
    "conference", "diabetes", "dmft", "fraud", "marketing", "monks1", "monks2", "monks3",
    "mushroom", "parity5", "phishing0", "reviewer", "sensory", "servo", "solar", "spect",
    "splice", "threeOf9", "ttt", "tumour", "vote", "xd6"
]

# # AUC ROC Values: OCC-CAT (AUCROC)
# # Baseline 1 Scores
# occ_cat_scores = np.array([
#     0.519, 0.536, 0.544, 0.560, 0.540, 0.919, 0.436, 0.813,
#     0.457, 0.618, 0.512, 0.511, 0.672, 0.671, 0.521, 0.561,
#     0.731, 0.536, 0.536, 0.472, 0.526, 0.529, 0.516, 0.616,
#     0.848, 0.477, 0.953, 0.420, 0.460, 0.607
# ])

# # Baseline 2 Scores: OCC-OHE-AE-RE (AUCROC)
# occ_ohe_ae_re_scores = np.array([
#     0.684, 0.624, 0.373, 0.532, 0.549, 0.722, 0.582, 0.679,
#     0.462, 0.336, 0.483, 0.577, 0.501, 0.722, 0.455, 0.666,
#     0.742, 0.494, 0.679, 0.564, 0.497, 0.595, 0.675, 0.285,
#     0.630, 0.768, 0.744, 0.730, 0.549, 0.770
# ])

# # New Proposed Classifier Scores: Top1 (AUCROC)
# top1_scores = np.array([
#     0.694, 0.650, 0.560, 0.786, 0.732, 0.915, 0.678, 0.813,
#     0.570, 0.648, 0.567, 0.606, 0.679, 0.738, 0.670, 0.813,
#     0.786, 0.588, 0.685, 0.688, 0.573, 0.657, 0.660, 0.809,
#     0.851, 0.823, 0.983, 0.764, 0.663, 0.866
# ])

# Baseline 1 Scores: OCC-CAT (AUCPR)
occ_cat_scores = np.array([
    0.931, 0.934, 0.791, 0.805, 0.683, 0.967, 0.745, 0.856,
    0.922, 0.832, 0.902, 0.890, 0.843, 0.754, 0.791, 0.706,
    0.779, 0.656, 0.709, 0.573, 0.703, 0.857, 0.914, 0.903,
    0.889, 0.622, 0.983, 0.764, 0.648, 0.847
])

# Baseline 2 Scores: OCC-OHE-AE-RE (AUCPR)
occ_ohe_ae_re_scores = np.array([
    0.916, 0.897, 0.678, 0.805, 0.577, 0.827, 0.747, 0.662,
    0.853, 0.503, 0.792, 0.919, 0.680, 0.704, 0.627, 0.631,
    0.760, 0.491, 0.740, 0.612, 0.571, 0.784, 0.898, 0.664,
    0.613, 0.786, 0.796, 0.882, 0.712, 0.838
])

# New Proposed Classifier Scores: Top1 (AUCPR)
top1_scores = np.array([
    0.939, 0.933, 0.848, 0.874, 0.786, 0.966, 0.856, 0.856,
    0.937, 0.832, 0.903, 0.949, 0.851, 0.758, 0.829, 0.784,
    0.852, 0.751, 0.779, 0.780, 0.747, 0.899, 0.918, 0.940,
    0.891, 0.815, 0.992, 0.908, 0.802, 0.915
])
print(f"Total number of verified datasets: {len(datasets)}")



Total number of verified datasets: 30


In [6]:
# ==========================================
# 2. STEP 1: OMNIBUS FRIEDMAN TEST
# ==========================================
friedman_stat, friedman_p = friedmanchisquare(occ_cat_scores, occ_ohe_ae_re_scores, top1_scores)

print("\n=== STEP 1: OMNIBUS FRIEDMAN TEST ===")
print(f"Friedman Chi-Square Statistic: {friedman_stat:.4f}")
print(f"Friedman asymptotic p-value:  {friedman_p:.5e}")

if friedman_p < 0.05:
    print("Result: Globally significant difference detected across the benchmarks! Proceeding to post-hoc analysis...")
else:
    print("Result: No globally significant difference detected at alpha=0.05.")




=== STEP 1: OMNIBUS FRIEDMAN TEST ===
Friedman Chi-Square Statistic: 42.9231
Friedman asymptotic p-value:  4.77939e-10
Result: Globally significant difference detected across the benchmarks! Proceeding to post-hoc analysis...


In [7]:
# ==========================================
# 3. STEP 2: POST-HOC PAIRWISE COMPARISONS (WILCOXON SIGNED-RANK)
# ==========================================
# Defining the three paired relationships
comparisons = [
    ("Top1 vs OCC-CAT", top1_scores, occ_cat_scores),
    ("Top1 vs OCC-OHE-AE-RE", top1_scores, occ_ohe_ae_re_scores),
    ("OCC-CAT vs OCC-OHE-AE-RE", occ_cat_scores, occ_ohe_ae_re_scores)
]

raw_p_values = []
comparison_labels = []

for label, group1, group2 in comparisons:
    # Running two-tailed Wilcoxon signed-rank test on paired dataset performance
    _, p_val = wilcoxon(group1, group2)
    raw_p_values.append(p_val)
    comparison_labels.append(label)

# ==========================================
# 4. STEP 3: HOLM-BONFERRONI MULTIPLE COMPARISON CORRECTION
# ==========================================
reject_null, adjusted_p_values, _, _ = multipletests(raw_p_values, alpha=0.05, method='holm')

print("\n=== STEP 2: POST-HOC PAIRWISE EVALUATION (WITH HOLM CORRECTION) ===")
for i in range(len(comparisons)):
    print(f"\nComparison Group: {comparison_labels[i]}")
    print(f"  -> Unadjusted (Raw) p-value: {raw_p_values[i]:.5e}")
    print(f"  -> Holm-Adjusted p-value:    {adjusted_p_values[i]:.5e}")
    print(f"  -> Reject Null Hypothesis (Significantly Different?): {reject_null[i]}")


=== STEP 2: POST-HOC PAIRWISE EVALUATION (WITH HOLM CORRECTION) ===

Comparison Group: Top1 vs OCC-CAT
  -> Unadjusted (Raw) p-value: 5.83593e-06
  -> Holm-Adjusted p-value:    1.16719e-05
  -> Reject Null Hypothesis (Significantly Different?): True

Comparison Group: Top1 vs OCC-OHE-AE-RE
  -> Unadjusted (Raw) p-value: 1.86265e-09
  -> Holm-Adjusted p-value:    5.58794e-09
  -> Reject Null Hypothesis (Significantly Different?): True

Comparison Group: OCC-CAT vs OCC-OHE-AE-RE
  -> Unadjusted (Raw) p-value: 1.98673e-03
  -> Holm-Adjusted p-value:    1.98673e-03
  -> Reject Null Hypothesis (Significantly Different?): True
